# Notebook 02 – Hypothesis Testing
**DSA 210 – Screen Time & Well-Being Project**

We test three hypotheses about the relationship between screen time and personal well-being indicators.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

sns.set_theme(style='whitegrid', font_scale=1.1)

df = pd.read_csv('../data/processed/merged_dataset.csv', parse_dates=['date'])
print(f'Loaded {len(df)} days of data.')
df.head(3)

---
## H1: Screen time is significantly higher on low-productivity days

**Groups:** Low productivity (score ≤ 5) vs High productivity (score > 5)  
**Test:** Two-sample independent t-test (+ Mann-Whitney U as non-parametric check)  
**Significance level:** α = 0.05

In [ ]:
low_prod  = df.loc[df['productivity_score'] <= 5, 'total_screen_time_min'].dropna()
high_prod = df.loc[df['productivity_score'] > 5,  'total_screen_time_min'].dropna()

print(f'Low productivity days  (n={len(low_prod)}): mean={low_prod.mean():.1f} min, std={low_prod.std():.1f}')
print(f'High productivity days (n={len(high_prod)}): mean={high_prod.mean():.1f} min, std={high_prod.std():.1f}')

# Normality check (Shapiro-Wilk)
_, p_norm_low  = stats.shapiro(low_prod)
_, p_norm_high = stats.shapiro(high_prod)
print(f'\nShapiro-Wilk p-values: low={p_norm_low:.3f}, high={p_norm_high:.3f}')

# Levene's test for equal variances
_, p_levene = stats.levene(low_prod, high_prod)
equal_var = p_levene > 0.05
print(f'Levene p={p_levene:.3f} → equal variances: {equal_var}')

# t-test
t_stat, p_ttest = stats.ttest_ind(low_prod, high_prod, equal_var=equal_var)
print(f'\nt-test: t={t_stat:.3f}, p={p_ttest:.4f}')

# Mann-Whitney U (non-parametric)
u_stat, p_mw = stats.mannwhitneyu(low_prod, high_prod, alternative='greater')
print(f'Mann-Whitney U: U={u_stat:.1f}, p={p_mw:.4f}')

# Effect size (Cohen's d)
pooled_std = np.sqrt((low_prod.std()**2 + high_prod.std()**2) / 2)
cohens_d = (low_prod.mean() - high_prod.mean()) / pooled_std
print(f"Cohen's d = {cohens_d:.3f}")

result = 'REJECTED' if p_ttest < 0.05 else 'NOT REJECTED'
print(f'\nH1 null hypothesis: {result} (α=0.05)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data_h1 = df.copy()
data_h1['Productivity Group'] = data_h1['productivity_score'].apply(
    lambda x: 'Low (≤5)' if x <= 5 else 'High (>5)'
)
sns.boxplot(data=data_h1, x='Productivity Group', y='total_screen_time_min',
            palette='Set2', ax=axes[0])
axes[0].set_title('H1: Screen Time by Productivity Group')
axes[0].set_ylabel('Total Screen Time (min)')

sns.scatterplot(data=df, x='productivity_score', y='total_screen_time_min',
                alpha=0.6, color='steelblue', ax=axes[1])
m, b = np.polyfit(df['productivity_score'].dropna(), df['total_screen_time_min'].dropna(), 1)
x_range = np.linspace(1, 10, 50)
axes[1].plot(x_range, m*x_range + b, 'r-', linewidth=2)
axes[1].set_title('Screen Time vs Productivity Score')

plt.tight_layout()
plt.savefig('../figures/h1_productivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## H2: High pickup count correlates with lower mood scores

**Test:** Pearson correlation + Spearman rank correlation  
**Alternative hypothesis:** negative correlation between pickups and mood

In [ ]:
valid = df[['pickups', 'mood_score']].dropna()
print(f'Valid rows: {len(valid)}')

# Pearson
r_pearson, p_pearson = stats.pearsonr(valid['pickups'], valid['mood_score'])
print(f'Pearson r = {r_pearson:.3f}, p = {p_pearson:.4f}')

# Spearman
r_spearman, p_spearman = stats.spearmanr(valid['pickups'], valid['mood_score'])
print(f'Spearman ρ = {r_spearman:.3f}, p = {p_spearman:.4f}')

result = 'SUPPORTED' if p_pearson < 0.05 and r_pearson < 0 else 'NOT SUPPORTED'
print(f'\nH2: Negative correlation hypothesis {result} (α=0.05)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=valid, x='pickups', y='mood_score', ax=ax,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
ax.set_title(f'H2: Pickups vs Mood (r={r_pearson:.2f}, p={p_pearson:.3f})')
plt.tight_layout()
plt.savefig('../figures/h2_pickups_mood.png', dpi=150, bbox_inches='tight')
plt.show()

---
## H3: Sleep duration negatively correlates with next-day screen time

**Test:** Pearson correlation between `sleep_hours` and `next_day_screen_time`

In [ ]:
# Create next-day screen time feature
df['next_day_screen_time'] = df['total_screen_time_min'].shift(-1)
valid_h3 = df[['sleep_hours', 'next_day_screen_time']].dropna()
print(f'Valid rows: {len(valid_h3)}')

r_h3, p_h3 = stats.pearsonr(valid_h3['sleep_hours'], valid_h3['next_day_screen_time'])
print(f'Pearson r = {r_h3:.3f}, p = {p_h3:.4f}')

r_h3_s, p_h3_s = stats.spearmanr(valid_h3['sleep_hours'], valid_h3['next_day_screen_time'])
print(f'Spearman ρ = {r_h3_s:.3f}, p = {p_h3_s:.4f}')

result = 'SUPPORTED' if p_h3 < 0.05 and r_h3 < 0 else 'NOT SUPPORTED'
print(f'\nH3: Negative sleep→screen time correlation {result} (α=0.05)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=valid_h3, x='sleep_hours', y='next_day_screen_time', ax=ax,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
ax.set_title(f'H3: Sleep Hours → Next-Day Screen Time (r={r_h3:.2f}, p={p_h3:.3f})')
ax.set_xlabel('Sleep Hours')
ax.set_ylabel('Next-Day Screen Time (min)')
plt.tight_layout()
plt.savefig('../figures/h3_sleep_screen.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary of Hypothesis Test Results

In [ ]:
summary = pd.DataFrame({
    'Hypothesis': [
        'H1: Screen time higher on low-productivity days',
        'H2: More pickups → lower mood',
        'H3: More sleep → less next-day screen time',
    ],
    'Test': ['t-test / Mann-Whitney U', 'Pearson/Spearman correlation', 'Pearson/Spearman correlation'],
    'p-value': [p_ttest, p_pearson, p_h3],
    'Effect Size': [f"Cohen's d = {cohens_d:.2f}", f'r = {r_pearson:.2f}', f'r = {r_h3:.2f}'],
    'Decision (α=0.05)': [
        'Reject H₀' if p_ttest < 0.05 else 'Fail to Reject H₀',
        'Reject H₀' if p_pearson < 0.05 else 'Fail to Reject H₀',
        'Reject H₀' if p_h3 < 0.05 else 'Fail to Reject H₀',
    ]
})
summary.style.applymap(
    lambda v: 'background-color: #d4edda' if 'Reject H₀' == v else '',
    subset=['Decision (α=0.05)']
)